# Exploratory Data Analysis (EDA) Project
**Dataset:** Titanic passengers (via seaborn)

**Goal:** Uncover patterns, correlations and key factors that influenced passenger survival.

> Tip: to use your own dataset, replace the loading cell with `df = pd.read_csv('your_file.csv')` and adapt the column names.

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
df = sns.load_dataset('titanic')
df.head()

## 2. Data Overview

In [ ]:
print('Shape:', df.shape)
df.info()

In [ ]:
df.describe(include='all').T

## 3. Data Quality: Missing Values & Duplicates

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing>0])
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
plt.figure(figsize=(9,4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Value Map')
plt.show()

### Handling missing values
- `age`: filled with the median (robust to outliers)
- `embarked` / `embark_town`: filled with the mode
- `deck`: dropped (~77% missing)

In [ ]:
df = df.drop_duplicates()
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])
df = df.drop(columns=['deck'])
print('Remaining missing:', df.isnull().sum().sum())

## 4. Statistical Summaries

In [ ]:
df[['age','fare','sibsp','parch']].describe()

In [ ]:
print('Survival rate:', round(df['survived'].mean()*100,1), '%')
for col in ['sex','pclass','embarked','alone']:
    print(f'\nSurvival rate by {col}:')
    print((df.groupby(col)['survived'].mean()*100).round(1))

## 5. Univariate Analysis

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,4))
sns.histplot(df['age'], kde=True, ax=ax[0]); ax[0].set_title('Age Distribution')
sns.histplot(df['fare'], kde=True, ax=ax[1]); ax[1].set_title('Fare Distribution')
sns.countplot(x='pclass', data=df, ax=ax[2]); ax[2].set_title('Passenger Class')
plt.tight_layout(); plt.show()

### Outlier detection

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,4))
sns.boxplot(y=df['age'], ax=ax[0]); ax[0].set_title('Age')
sns.boxplot(y=df['fare'], ax=ax[1]); ax[1].set_title('Fare')
plt.show()

Q1, Q3 = df['fare'].quantile([.25,.75]); IQR = Q3-Q1
print('Fare outliers (IQR rule):', ((df['fare']<Q1-1.5*IQR)|(df['fare']>Q3+1.5*IQR)).sum())

## 6. Bivariate Analysis: What influenced survival?

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(15,4))
sns.barplot(x='sex', y='survived', data=df, ax=ax[0]); ax[0].set_title('Survival by Sex')
sns.barplot(x='pclass', y='survived', data=df, ax=ax[1]); ax[1].set_title('Survival by Class')
sns.barplot(x='embarked', y='survived', data=df, ax=ax[2]); ax[2].set_title('Survival by Port')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.kdeplot(data=df, x='age', hue='survived', fill=True, common_norm=False)
plt.title('Age Distribution by Survival'); plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(x='pclass', y='survived', hue='sex', data=df)
plt.title('Survival by Class and Sex'); plt.show()

## 7. Correlation Analysis

In [ ]:
num = df.select_dtypes(include=np.number)
plt.figure(figsize=(8,6))
sns.heatmap(num.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap'); plt.show()

In [ ]:
sns.pairplot(df[['survived','age','fare','pclass']], hue='survived', corner=True)
plt.show()

## 8. Key Insights & Report
1. **Sex** was the strongest factor: women survived at a far higher rate than men.
2. **Passenger class** mattered: 1st class had the highest survival rate, 3rd class the lowest.
3. **Fare** is positively correlated with survival (a proxy for class).
4. **Age**: children had better survival chances; older passengers fared worse.
5. **Travelling alone** was associated with lower survival than small families.
6. **Data quality:** `age` had ~20% missing values (median-imputed); `deck` was dropped due to ~77% missing.

### Conclusion
Survival was shaped mainly by sex, class and age, consistent with the 'women and children first' protocol and unequal access to lifeboats by class.

*(Check the numbers in your own run and update this section to match your output.)*